# RAG with Knowledge Graph (Neo4j)

This notebook builds a Retrieval-Augmented Generation pipeline using Neo4j as the knowledge graph backend.

**Stack:**
- LLM: OpenRouter (Llama 3.1 70B) via LangChain
- Embeddings: HuggingFace `BAAI/bge-small-en-v1.5` (free, local)
- Graph DB: Neo4j AuraDB
- Data Source: Wikipedia

## Step 1 — Install Dependencies

In [1]:
%pip install --upgrade --quiet \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-experimental \
    langchain-huggingface \
    langchain-neo4j \
    langchain-text-splitters \
    neo4j \
    wikipedia \
    tiktoken \
    sentence-transformers==2.7.0 \
    python-dotenv \
    yfiles_jupyter_graphs

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Load Environment Variables

Create a `.env` file in the same directory with the following keys:
```
OPENROUTER_API_KEY=your_key_here
NEO4J_URI=neo4j+s://xxxx.databases.neo4j.io
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password_here
```

In [2]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API Key: ")

if "NEO4J_URI" not in os.environ:
    os.environ["NEO4J_URI"] = input("Enter Neo4j URI: ")

if "NEO4J_USERNAME" not in os.environ:
    os.environ["NEO4J_USERNAME"] = input("Enter Neo4j Username: ")

if "NEO4J_PASSWORD" not in os.environ:
    os.environ["NEO4J_PASSWORD"] = getpass.getpass("Enter Neo4j Password: ")

print("All credentials loaded.")

All credentials loaded.


## Step 3 — Initialize LLM and Embeddings

- LLM: Llama 3.1 70B via OpenRouter (free tier available)
- Embeddings: `BAAI/bge-small-en-v1.5` runs locally, no API cost

In [3]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

# Free local embeddings
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# OpenRouter LLM
llm = ChatOpenAI(
    model="meta-llama/llama-3.1-70b-instruct",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

print("Embeddings model loaded: BAAI/bge-small-en-v1.5")
print("LLM initialized: Llama 3.1 70B via OpenRouter")

c:\Users\Radhakrishna\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Embeddings model loaded: BAAI/bge-small-en-v1.5
LLM initialized: Llama 3.1 70B via OpenRouter


## Step 4 — Connect to Neo4j Graph

In [4]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"]
)

print("Connected to Neo4j.")

Connected to Neo4j.


## Step 5 — Load and Split Documents

In [7]:
from langchain_core.documents import Document

raw_documents = [
    Document(
        page_content="""Elizabeth I (7 September 1533 – 24 March 1603) was Queen of England and Ireland from 17 November 1558 until her death in 1603. 
        She was the last monarch of the House of Tudor. Elizabeth was the daughter of Henry VIII and his second wife, Anne Boleyn. 
        Her mother was executed when Elizabeth was two years old, and she was declared illegitimate. 
        Her half-brother Edward VI ruled until his death in 1553, bequeathing the crown to Lady Jane Grey. 
        His will was set aside, and Elizabeth's half-sister Mary I took the throne before dying childless in 1558, 
        whereupon Elizabeth succeeded her.""",
        metadata={"title": "Elizabeth I", "source": "manual"}
    ),
    Document(
        page_content="""Elizabeth's reign is known as the Elizabethan era. During this time, the arts flourished and 
        England asserted itself as a major European power. Her reign saw the defeat of the Spanish Armada in 1588, 
        a major victory against Spain. Sir Francis Drake, Sir Walter Raleigh, and other explorers expanded English 
        influence across the globe. Elizabeth never married and was known as the Virgin Queen. 
        She was succeeded by her cousin James VI of Scotland, who became James I of England.""",
        metadata={"title": "Elizabeth I - Reign", "source": "manual"}
    ),
    Document(
        page_content="""Elizabeth I was born at Greenwich Palace and was the daughter of King Henry VIII of England 
        and his second wife, Anne Boleyn. Elizabeth was educated by tutors and became highly educated for her time, 
        speaking several languages including Latin, Greek, French, and Italian. 
        Robert Dudley, Earl of Leicester, was one of her closest companions throughout her life. 
        William Cecil, 1st Baron Burghley, served as her chief advisor for much of her reign. 
        Elizabeth died at Richmond Palace on 24 March 1603 at the age of 69.""",
        metadata={"title": "Elizabeth I - Personal Life", "source": "manual"}
    ),
]

print(f"Loaded {len(raw_documents)} documents")

Loaded 3 documents


In [8]:
raw_documents[:3]

[Document(metadata={'title': 'Elizabeth I', 'source': 'manual'}, page_content="Elizabeth I (7 September 1533 – 24 March 1603) was Queen of England and Ireland from 17 November 1558 until her death in 1603. \n        She was the last monarch of the House of Tudor. Elizabeth was the daughter of Henry VIII and his second wife, Anne Boleyn. \n        Her mother was executed when Elizabeth was two years old, and she was declared illegitimate. \n        Her half-brother Edward VI ruled until his death in 1553, bequeathing the crown to Lady Jane Grey. \n        His will was set aside, and Elizabeth's half-sister Mary I took the throne before dying childless in 1558, \n        whereupon Elizabeth succeeded her."),
 Document(metadata={'title': 'Elizabeth I - Reign', 'source': 'manual'}, page_content="Elizabeth's reign is known as the Elizabethan era. During this time, the arts flourished and \n        England asserted itself as a major European power. Her reign saw the defeat of the Spanish Arm

In [9]:
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])
print(f"Split into {len(documents)} chunks")

Split into 3 chunks


## Step 6 — Extract Graph Documents using LLM

`LLMGraphTransformer` uses the LLM to extract entities and relationships from text and converts them into graph-compatible documents.

In [10]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(llm=llm)
graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(f"Converted {len(graph_documents)} graph documents")

C:\Users\Radhakrishna\AppData\Local\Temp\ipykernel_41524\3514095672.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


Converted 3 graph documents


In [11]:
graph_documents

[GraphDocument(nodes=[Node(id='Elizabeth I', type='Person', properties={}), Node(id='House Of Tudor', type='Dynasty', properties={}), Node(id='Henry Viii', type='Person', properties={}), Node(id='Anne Boleyn', type='Person', properties={}), Node(id='Edward Vi', type='Person', properties={}), Node(id='Lady Jane Grey', type='Person', properties={}), Node(id='Mary I', type='Person', properties={}), Node(id='England', type='Country', properties={}), Node(id='Ireland', type='Country', properties={})], relationships=[Relationship(source=Node(id='Elizabeth I', type='Person', properties={}), target=Node(id='House Of Tudor', type='Dynasty', properties={}), type='MEMBER_OF', properties={}), Relationship(source=Node(id='Elizabeth I', type='Person', properties={}), target=Node(id='Henry Viii', type='Person', properties={}), type='CHILD_OF', properties={}), Relationship(source=Node(id='Elizabeth I', type='Person', properties={}), target=Node(id='Anne Boleyn', type='Person', properties={}), type='CH

## Step 7 — Store Graph Documents in Neo4j

In [12]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)
print("Graph documents stored in Neo4j.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=257, offset=256>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 256, 'line': 1, 'column': 257}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MERGE (d:Document {id:$document.metadata.id}) SET d.text = $document.page_content SET d += $document.metadata WITH d UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties MERGE (d)-[:MENTIONS]->(source) WITH source, row CALL apoc.create.addLabels( source, [row.type] 

Graph documents stored in Neo4j.


## Step 8 — Visualize the Knowledge Graph

Uses `yfiles_jupyter_graphs` to render the graph inline in the notebook.

In [13]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

def showGraph(cypher: str = default_cypher):
    driver = GraphDatabase.driver(
        uri=os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
    )
    session = driver.session()
    widget = GraphWidget(graph=session.run(cypher).graph())
    widget.node_label_mapping = "id"
    display(widget)
    return widget

showGraph()

GraphWidget(layout=Layout(height='720px', width='100%'))

GraphWidget(layout=Layout(height='720px', width='100%'))

## Step 9 — Build Vector Index for Hybrid Search

Creates a Neo4j vector index using HuggingFace embeddings. Enables semantic similarity search alongside graph traversal.

In [14]:
from langchain_neo4j import Neo4jVector

vector_index = Neo4jVector.from_existing_graph(
    embedding,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)

print("Vector index created.")

Vector index created.


## Step 10 — Create Fulltext Index for Entity Search

In [15]:
graph.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")
print("Fulltext index created.")

Fulltext index created.


## Step 11 — Define Entity Extraction Model

Pydantic model to extract named entities (persons, organizations) from a user question.

In [16]:
from typing import Tuple, List, Optional
from pydantic import BaseModel, Field

class Entities(BaseModel):
    """Identifying information about entities."""
    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that appear in the text",
    )

## Step 12 — Build Entity Extraction Chain

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

entity_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are extracting organization and person entities from the text."),
        ("human", "Use the given format to extract information from the following input: {question}"),
    ]
)

entity_chain = entity_prompt | llm.with_structured_output(Entities)

In [18]:
# Test entity extraction
entity_chain.invoke({"question": "Where was Amelia Earhart born?"}).names

['Amelia Earhart']

## Step 13 — Build Structured Graph Retriever

Traverses the knowledge graph using fulltext entity search and relationship traversal.

In [19]:
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

def generate_full_text_query(input: str) -> str:
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()


def structured_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node, score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el["output"] for el in response])
    return result


# Test structured retriever
print(structured_retriever("Who is Elizabeth I?"))

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

Elizabeth I - MEMBER_OF -> House Of Tudor
Elizabeth I - CHILD_OF -> Henry Viii
Elizabeth I - CHILD_OF -> Anne Boleyn
Elizabeth I - SIBLING_OF -> Edward Vi
Elizabeth I - SIBLING_OF -> Mary I
Elizabeth I - MONARCH_OF -> England
Elizabeth I - MONARCH_OF -> Ireland
Elizabeth I - PARENT -> Anne Boleyn
Elizabeth I - PARENT -> King Henry Viii Of England
Elizabeth I - COMPANION -> Robert Dudley, Earl Of Leicester
Elizabeth I - ADVISOR -> William Cecil, 1St Baron Burghley
Elizabeth I - BIRTHPLACE -> Greenwich Palace
Elizabeth I - DEATHPLACE -> Richmond Palace
Mary I - PREDECEDED_BY -> Elizabeth I


## Step 14 — Build Hybrid Retriever

Combines structured graph retrieval (entity relationships) with unstructured vector similarity search.

In [20]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ".join(unstructured_data)}
    """
    return final_data

## Step 15 — Build Conversational RAG Chain

Handles multi-turn conversations by condensing chat history into a standalone question before retrieval.

In [21]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

_search_query = RunnableBranch(
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | llm
        | StrOutputParser(),
    ),
    RunnableLambda(lambda x: x["question"]),
)

template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built.")

RAG chain built.


## Step 16 — Run Queries

In [22]:
# Single question
chain.invoke({"question": "Which house did Elizabeth I belong to?"})

Search query: Which house did Elizabeth I belong to?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

'Elizabeth I belonged to the House of Tudor.'

In [23]:
# Multi-turn with chat history
chain.invoke(
    {
        "question": "When was she born?",
        "chat_history": [("Which house did Elizabeth I belong to?", "House Of Tudor")],
    }
)

Search query: When was Elizabeth I born?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

'Elizabeth I was born on 7 September 1533.'